# Develop a Multi-Agent Research Assistant Using AutoGen

### Task 0: Set up Your Environment and Load API Keys

In [ ]:
from dotenv import load_dotenv
import os
load_dotenv()

In [ ]:
from autogen_ext.models.openai import OpenAIChatCompletionClient
model_client = OpenAIChatCompletionClient(
    model="gpt-5-nano",
    api_key=os.getenv("OPENAI_API_KEY")
)

### Task 1: Import Libraries

In [ ]:
from typing import List, Sequence

from autogen_agentchat.agents import AssistantAgent, UserProxyAgent
from autogen_agentchat.conditions import MaxMessageTermination, TextMentionTermination
from autogen_agentchat.messages import BaseAgentEvent, BaseChatMessage
from autogen_agentchat.teams import SelectorGroupChat, RoundRobinGroupChat
from autogen_agentchat.ui import Console
from autogen_ext.models.openai import OpenAIChatCompletionClient
from dotenv import load_dotenv
import requests
import xml.etree.ElementTree as ET

### Task 2: Implement an arXiv Paper Search Function

In [ ]:
def arxiv_search_tool(query:str, max_results:int=3) -> str:
    """
    Search papers from arXiv related to the given query.
    Returns a human-readable summary of results.
    """
    base_url = "http://export.arxiv.org/api/query"
    params = {
        "search_query": f"all:{query}",
        "start": 0,
        "max_results": max_results
    }
    response = requests.get(base_url, params=params, timeout=10)
    if response.status_code != 200:
        return f"Error: Unable to fetch data from arXiv (status {response.status_code})"

    # Parse the XML response
    root = ET.fromstring(response.text)
    ns = {"atom": "http://www.w3.org/2005/Atom"}
    entries = root.findall("atom:entry", ns)
    
    if not entries:
        return f"No results being found for {query}"
    
    output = [f"Papers for '{query}':"]
    for i, entry in enumerate(entries, start=1):
        title = entry.find("atom:title", ns).text.strip()
        summary = entry.find("atom:summary", ns).text.strip().replace("\n", " ")
        link = entry.find("atom:id", ns).text.strip()
        output.append(f"{i}. {title}\n   Link: {link}\n   Abstract: {summary[:300]}...")
    
    return "\n".join(output)


### Task 3: Create the Topic Refinement Agent

In [ ]:
# Create the topic refinement agent
topic_agent = AssistantAgent(
    "TopicAgent",
    description="Refines a broad research idea into a specific topic.",
    model_client=model_client,
    system_message="""
    ROLE: You are a Research Topic Refiner.
    TASK: Given a general idea, output one refined research topic.

    OUTPUT FORMAT:
    Refined Research Topic: <Topic>
    End with: TASK_COMPLETED_TOPIC_AGENT
    """
)

In [ ]:
topic_result = await topic_agent.run(task="AI in healthcare")
print(topic_result.messages[-1].content)

### Task 4: Create the Paper Discovery Agent

In [ ]:
paper_agent = AssistantAgent(
    "PaperAgent",
    tools = [arxiv_search_tool],
    description="Search research papers related to a given topic",
    model_client=model_client,
    system_message="""
    ROLE: You are a Paper Discovery Agent.
    TASK: Find 2–3 recent research papers related to the topic..

    OUTPUT FORMAT:
    Refined Research Topic: <Topic>
    - Paper 1: <Title> — <Summary> - <URL>
    - Paper 2: <Title> — <Summary> - <URL>
    """
)

In [ ]:
paper_result = await paper_agent.run(task=topic_result.messages[-1].content)
print(paper_result.messages[-1].content)

### Task 5: Add the Insight Synthesizer Agent

In [ ]:
insight_agent = AssistantAgent(
    "InsightAgent",
    description="summarizes keys insights from discovered paper",
    model_client=model_client,
    system_message="""
    ROLE: You are a Insight Synthesizer Agent.
    TASK: Produce 3-5 consies bullet points summarizing common findings

    OUTPUT FORMAT:
    Key Insights:
    - <Insight 1>
    - <Insight 2>
    """
)

In [ ]:
insight_result = await insight_agent.run(task=paper_result.messages[-1].content)
print(insight_result.messages[-1].content)

### Task 6: Create the Report Compiler Agent

In [ ]:
report_agent = AssistantAgent(
    "ReportAgent",
    description="to compile a concise research report",
    model_client=model_client,
    system_message="""
    ROLE: You are a Report Compiler Agent.
    TASK: Create a mini research report

    OUTPUT FORMAT:
    ## Research Report
    ### Introduction
    ...
    ### Key Findings
    ...
    ### Discussion
    ...
    ### Conclusion
    ...
    """
)

In [ ]:
report_result = await report_agent.run(task=insight_result.messages[-1].content)
print(report_result.messages[-1].content)

### Task 7: Add the Gap Analysis Agent

In [ ]:
gap_agent = AssistantAgent(
    "GaptAgent",
    description="to perform reseach gap analysis and propose next steps",
    model_client=model_client,
    system_message="""
    ROLE: You are a Research Gap Analyst.
    TASK: Output 1-2 research gaps and 1-2 future directions

    Research Gaps:
    - <Gap 1>
    - <Gap 2>

    Future Directions:
    - <Direction 1>
    - <Direction 2>

    FINAL REPORT
    """
)

In [ ]:
gap_result = await gap_agent.run(task=report_result.messages[-1].content)
print(gap_result.messages[-1].content)

### Task 8: Define Termination Conditions

In [ ]:
text_mention_termination = TextMentionTermination("FINAL REPORT")
max_messages_termination = MaxMessageTermination(max_messages=10)
termination = max_messages_termination | text_mention_termination

### Task 9: Build the Multi-Agent Workflow

In [ ]:
team = RoundRobinGroupChat(
    [topic_agent, paper_agent, insight_agent, report_agent, gap_agent], 
    termination_condition=termination
    )
await Console(team.run_stream(task="AI for Economy"))

### Task 10: Insert a User Proxy Agent for Manual Approval

In [ ]:
user_proxy_agent = UserProxyAgent(
    "UserProxyAgent",
    description="A proxy for the user to approve or disapprove tasks."
)

### Task 11: Implement a Custom Selector Function for User Approval

In [ ]:
user_approved = False
def selector_func_with_user_proxy(messages):
    global user_approved

    # If first message, start with topic agent
    if len(messages) == 0:
        return topic_agent.name

    last_source = messages[-1].source

    if last_source == topic_agent.name and not user_approved:
        return user_proxy_agent.name

    # If user just responded (approve or reject)
    if last_source == user_proxy_agent.name:
        content = messages[-1].content.upper()
        if "APPROVE" in content:
            user_approved = True
            # Proceed to the next logical agent (e.g., paper_agent)
            return paper_agent.name
        else:
            # If disapproved, return to topic_agent for revision
            return topic_agent.name
    
    # After approval, let the rest of the agents talk automatically
    if user_approved:
        return None  # means use default sequencing in SelectorGroupChat

    return None



### Task 12: Execute the Full Interactive Research Workflow

In [ ]:
selector_prompt = """Select an agent to perform task.

{roles}

Current conversation context:
{history}

Read the above conversation, then select an agent from {participants} to perform the next task.
Make sure the topic agent has assigned tasks before other agents start working.
Only select one agent.
"""

task = "AI for Data Engineer"


# Run the chat again with the user proxy agent and selector function.
team = SelectorGroupChat(
    [topic_agent, paper_agent, insight_agent, report_agent, gap_agent, user_proxy_agent],
    model_client=model_client,
    termination_condition=termination,
    selector_prompt=selector_prompt,
    selector_func=selector_func_with_user_proxy,
    allow_repeated_speaker=False,
)

await Console(team.run_stream(task=task))